# Spin Polarization, Antiferromagnetism, and Spin–Orbit Coupling in SIESTA — Theory Summary
---
---

In Density Functional Theory, **spin is an electronic property**, not an atomic one. Magnetism arises from an imbalance between spin-up and spin-down electron densities, defined as  
                                       $$ m(r) = \rho_{\uparrow}(r) - \rho_{\downarrow}(r) $$

Therefore, magnetic moments come from the **electron density distribution in space**, not from atoms themselves. However, codes like SIESTA use **atomic-centered basis orbitals**, so the initial spin configuration is conveniently specified *per atom* using `DM.InitSpin`. This does not mean atoms physically carry spin; it simply biases the **initial guess of the electronic density** near each atom so that the SCF cycle can converge to the desired magnetic solution. Without breaking this symmetry, a calculation may incorrectly remain non-magnetic even for a truly magnetic material. After convergence, the “atomic magnetic moments” reported are obtained from **Mulliken projections**, which are just integrals of the spin density around each atom for analysis purposes.

For magnetic phases, the number of atoms in the simulation cell determines what spin orderings can be represented. A primitive fcc cell contains only one atom and can describe only non-magnetic or ferromagnetic states. **Antiferromagnetism requires at least two atoms** so opposite spins can be initialized on different sublattices (↑ and ↓). Thus we use the smallest possible supercell (two atoms) to model AFM efficiently. When spin–orbit coupling (SOC) is included, relativity causes a moving electron in the nuclear electric field to experience an effective magnetic field, which couples its spin and orbital angular momentum (L·S interaction). This mixes spin-up and spin-down states, requiring spinor wavefunctions and leading to effects such as band splitting and magnetic anisotropy. In practice, magnetic simulations follow three key ideas: enable spin polarization, break symmetry with an initial spin guess, and verify the final magnetic state by comparing total energies and Mulliken spin moments.


In [1]:
!pwd

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/02_Fe_fcc


In [2]:
# create directory for exercise only 
!mkdir -p EX1

In [3]:
ls

 EX1/   fe_fcc.fdf   Fe.psf  'Tutorial SPIN2.ipynb'


In [4]:
!cp fe_fcc.fdf   Fe.psf EX1/ 

In [5]:
%cd EX1

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/02_Fe_fcc/EX1


/home/l-rishiraj/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
ls

fe_fcc.fdf  Fe.psf


In [7]:
# display the fdf.file
!cat fe_fcc.fdf

#General system specifications
SystemName          Iron (fcc) - Antiferromagnetic
SystemLabel         fe_fcc
NumberOfAtoms       2
NumberOfSpecies     1

%block ChemicalSpeciesLabel
 1  26 Fe      # Species index, atomic number, species label
%endblock ChemicalSpeciesLabel

# Basis set definition
PAO.EnergyShift 200 meV
PAO.SplitNorm   0.15
PAO.BasisSize   DZP

# Lattice vectors
LatticeConstant  3.66 Ang
%block LatticeVectors
    0.5000   -0.5000    0.0000
    0.5000    0.5000    0.0000
    0.0000    0.0000    1.0000
%endblock LatticeVectors

#Atomic coordinates
AtomicCoordinatesFormat  ScaledCartesian

%block AtomicCoordinatesAndAtomicSpecies
    0.0000    0.0000    0.0000   1           
    0.5000    0.0000    0.5000   1           
%endblock AtomicCoordinatesAndAtomicSpecies

# Real space grid 
MeshCutoff 125.0 Ry

# K points
%block kgrid.MonkhorstPack
  4    0    0    0.5
  0    4    0    0.5
  0    0    4    0.5
%endblock kgrid.MonkhorstPack

# Smearing
OccupationFunction    MP
Occ

---
From the fdf file, we notice that **Spin option** is not take part for the simulation by default in siesta. So, we need to turn on these section to **run antiferromagnetic materials and give accurate reasons at siesta**.
Therefore, it this section is change **by default** into like as follow as per given in the tutorial materials:
           
```
# Spin options
Spin polarized

%block DM.InitSpin
  1  +
  2  -
%endblock DM.InitSpin
```





---

In [34]:
# Now run the siesta

!siesta < fe_fcc.fdf > fe_fcc.out


Job completed


In [35]:
ls

0_NORMAL_EXIT                fe_fcc.DM          Fe.ion.nc
BASIS_ENTHALPY               fe_fcc.EIG         Fe.ion.xml
BASIS_HARRIS_ENTHALPY        fe_fcc.FA          Fe.psf
CLOCK                        fe_fcc.fdf         FORCE_STRESS
fdf.20260209T110759.842.log  fe_fcc.HSX         INPUT_TMP.38027
fdf.20260209T111746.857.log  fe_fcc.KP          INPUT_TMP.51012
fe_fcc.alloc                 fe_fcc.ORB_INDX    MESSAGES
fe_fcc.BASIS_ENTHALPY        fe_fcc.out         NON_TRIMMED_KP_LIST
fe_fcc.bib                   fe_fcc.STRUCT_OUT  OUTVARS.yml
fe_fcc.BONDS                 fe_fcc.XV          PARALLEL_DIST
fe_fcc.BONDS_FINAL           Fe.ion


In [36]:
!grep  -A 20 mulliken fe_fcc.out


mulliken: Atomic and Orbital Populations:

mulliken: Spin UP 

Species: Fe                  
Atom  Qatom  Qorb
               4s      4s      3dxy    3dyz    3dz2    3dxz    3dx2-y2 3dxy    
               3dyz    3dz2    3dxz    3dx2-y2 4Ppy    4Ppz    4Ppx    
   1  5.111  -0.040   0.270   0.810   0.844   0.871   0.844   0.898  -0.019
             -0.025  -0.015  -0.025  -0.014   0.229   0.254   0.229
   2  2.889  -0.102   0.267   0.613   0.489   0.459   0.489   0.381  -0.067
             -0.053  -0.050  -0.053  -0.047   0.192   0.180   0.192

mulliken: Qtot =        8.000

mulliken: Spin DOWN 

Species: Fe                  
Atom  Qatom  Qorb
               4s      4s      3dxy    3dyz    3dz2    3dxz    3dx2-y2 3dxy    
               3dyz    3dz2    3dxz    3dx2-y2 4Ppy    4Ppz    4Ppx    
   1  2.889  -0.102   0.267   0.613   0.489   0.459   0.489   0.381  -0.067
             -0.053  -0.050  -0.053  -0.047   0.192   0.180   0.192
   2  5.111  -0.040   0.270   0.810   0.844   0.871

---
# Magnetic Analysis of fcc Fe (SIESTA)

## Q1. What is the final spin magnetic moment in the unit cell? On each Fe atom?

### Answer

From the Mulliken spin populations:

- Atom 1 → **+2.22 μB**
- Atom 2 → **−2.22 μB**

Total:

$[
M_{\text{cell}} = (+2.22) + (-2.22) = 0 \, \mu_B
]$

So:

- **Unit cell moment = 0 μB**
- **Each Fe atom ≈ 2.2 μB (opposite directions)**

---

## Q2. Does the final magnetic phase match what you expected?

### Answer

Yes.

The spins were initialized as: 

                  
                   1  +
                   2  -
                   


which corresponds to antiparallel alignment (antiferromagnetic).

The results show equal and opposite moments with zero total magnetization, confirming an **Antiferromagnetic (AFM)** ground state.

---

## Q3. Compare the antiferromagnetic structure to the ferromagnetic one. In which phase are the magnetic moments larger?

### Answer

### Antiferromagnetic (AFM)
- +2.22 μB
- −2.22 μB
- Total = 0

### Ferromagnetic (FM)
- +2.3 μB
- +2.3 μB
- Total ≈ 4.6 μB

Therefore:

$[
|M_{\text{FM}}| > |M_{\text{AFM}}|
]$

The **ferromagnetic phase has slightly larger local magnetic moments** due to stronger exchange splitting.
                   
---


In [26]:
!tail fe_fcc.out


overfsm                1       0.004       0.004     0.05
writeHSX               1       0.001       0.001     0.01
state_analysis         1       0.000       0.000     0.00
siesta_move            1       0.000       0.000     0.00
Analysis               1       0.002       0.002     0.02
optical                1       0.000       0.000     0.00
  

>> End of run:   9-FEB-2026  11:08:04
Job completed


In [28]:
!grep -i spin fe_fcc.out | head


# Spin options
SpinPolarized  true
%block DM.InitSpin
%endblock DM.InitSpin
NOTE on units for spin output: 
{S} is the spin magnetic moment, in units of the Bohr magneton,
assuming a g-factor for the electron spin of exactly 2.
The spin magnetic moment is then numerically equal to the charge inbalance,
in units of electrons, between the spin-up and -down channels.
There are spin-orbit semi-local pseudopotentials available


In [27]:
!grep -i moment fe_fcc.out


{S} is the spin magnetic moment, in units of the Bohr magneton,
The spin magnetic moment is then numerically equal to the charge inbalance,
SPLIT: Orbitals with angular momentum L= 0
SPLIT: Orbitals with angular momentum L= 2
     spin moment: {S} , |S| = {        0.0        0.0   -0.00000 }    -0.00000
     spin moment: {S} , |S| = {        0.0        0.0   -0.00000 }    -0.00000
     spin moment: {S} , |S| = {        0.0        0.0   -0.00000 }    -0.00000
     spin moment: {S} , |S| = {        0.0        0.0    0.00000 }     0.00000
     spin moment: {S} , |S| = {        0.0        0.0   -0.00000 }    -0.00000
     spin moment: {S} , |S| = {        0.0        0.0   -0.00000 }    -0.00000
     spin moment: {S} , |S| = {        0.0        0.0   -0.00000 }    -0.00000
     spin moment: {S} , |S| = {        0.0        0.0    0.00000 }     0.00000
     spin moment: {S} , |S| = {        0.0        0.0    0.00000 }     0.00000
     spin moment: {S} , |S| = {        0.0        0.0   -0.0000

# EX2

In [51]:
%cd ..

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/02_Fe_fcc


/home/l-rishiraj/miniconda3/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [52]:
!mkdir -p EX2

In [53]:
!cp fe_fcc.fdf   Fe.psf EX2/

In [54]:
%cd EX2

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/02_Fe_fcc/EX2


In [68]:
!siesta < fe_fcc.fdf >fe_fcc.out

Job completed


In [59]:
!grep -i InitSpin fe_fcc.out

%block DM.InitSpin
%endblock DM.InitSpin


In [69]:
!grep  -A 20 mulliken fe_fcc.out


mulliken: Atomic and Orbital Populations:

mulliken: Spin UP 

Species: Fe                  
Atom  Qatom  Qorb
               4s      4s      3dxy    3dyz    3dz2    3dxz    3dx2-y2 3dxy    
               3dyz    3dz2    3dxz    3dx2-y2 4Ppy    4Ppz    4Ppx    
   1  5.111  -0.040   0.270   0.810   0.844   0.871   0.844   0.898  -0.019
             -0.025  -0.015  -0.025  -0.014   0.229   0.254   0.229
   2  2.889  -0.102   0.267   0.613   0.489   0.459   0.489   0.381  -0.067
             -0.053  -0.050  -0.053  -0.047   0.192   0.180   0.192

mulliken: Qtot =        8.000

mulliken: Spin DOWN 

Species: Fe                  
Atom  Qatom  Qorb
               4s      4s      3dxy    3dyz    3dz2    3dxz    3dx2-y2 3dxy    
               3dyz    3dz2    3dxz    3dx2-y2 4Ppy    4Ppz    4Ppx    
   1  2.889  -0.102   0.267   0.613   0.489   0.459   0.489   0.381  -0.067
             -0.053  -0.050  -0.053  -0.047   0.192   0.180   0.192
   2  5.111  -0.040   0.270   0.810   0.844   0.871

In [63]:
ls

0_NORMAL_EXIT                fe_fcc.BONDS_FINAL  Fe.ion.nc
BASIS_ENTHALPY               fe_fcc.DM           Fe.ion.xml
BASIS_HARRIS_ENTHALPY        fe_fcc.EIG          Fe.psf
CLOCK                        fe_fcc.FA           FORCE_STRESS
fdf.20260209T112121.353.log  fe_fcc.fdf          INPUT_TMP.52523
fdf.20260209T112518.281.log  fe_fcc.HSX          INPUT_TMP.76772
fdf.20260209T112645.602.log  fe_fcc.KP           INPUT_TMP.89451
fdf.20260209T114205.517.log  fe_fcc.ORB_INDX     INPUT_TMP.96687
fe_fcc.alloc                 fe_fcc.out          MESSAGES
fe_fcc.BASIS_ENTHALPY        fe_fcc.STRUCT_OUT   NON_TRIMMED_KP_LIST
fe_fcc.bib                   fe_fcc.XV           OUTVARS.yml
fe_fcc.BONDS                 Fe.ion              PARALLEL_DIST


In [65]:
!pwd

/home/l-rishiraj/siesta-docs/work-files/tutorials/basic/magnetism/02_Fe_fcc/EX2
